# RAGAS Filter Pipeline — End-to-End Demo

**Objective:** Demonstrate the full pipeline on branch `final-filtering-pipeline`:

```
labeled corpus → RAGAS features → train sklearn filter → apply to RAG predictions
```

**Dataset:** `data/merged/labeled_merged.csv` (training features) and `results/normal_rag/merged/predictions.csv` (RAG output to filter).

**Prerequisites**
1. Python venv with dependencies: `pip install -r requirements.txt`
2. `OPENAI_API_KEY` in a repo-root `.env` file (RAGAS feature extraction uses OpenAI)
3. For Stage 3, either existing normal-RAG predictions or the notebook builds a small demo predictions file

**Cost control:** Set `DEMO_LIMIT` below (e.g. `20`) before running OpenAI-backed cells.

## 0. Setup

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

# Resolve repo root whether the kernel cwd is /notebooks or repo root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rag_filtering.config.loader import load_yaml, resolve_path
from rag_filtering.evaluation.ragas_wrapper import RAGAS
from rag_filtering.filtering.filter_evaluator import select_threshold_min_fpr
from rag_filtering.filtering.ragas_feature_extractor import RagasFeatureExtractor
from rag_filtering.filtering.ragas_filter import RagasFilter
from rag_filtering.filtering.ragas_filter_trainer import RagasFilterTrainer

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

CONFIG_PATH = "configs/experiments/ragas_filter.yaml"
cfg = load_yaml(CONFIG_PATH)

# Cap OpenAI/RAGAS calls for demos. Set to None for a full run.
DEMO_LIMIT = 20
SEED = cfg.get("seed", 42)

load_dotenv(PROJECT_ROOT / ".env")
print("Project root:", PROJECT_ROOT)
print("Config loaded:", resolve_path(CONFIG_PATH))
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

## 1. (Optional) Generate normal-RAG predictions

Stage 3 expects `results/normal_rag/merged/predictions.csv`. If you have not run normal RAG yet, use the CLI (GPU + HuggingFace model download required):

```powershell
# From repo root, with venv activated
python experiments/normal_rag_inference/run_inference.py `
  --config configs/experiments/normal_rag_merged.yaml `
  --limit 20
```

The cell below creates a **small demo predictions file** from the labeled corpus when real RAG output is missing, so you can still test Stage 3 without loading the generator.

In [ ]:
rag_predictions_csv = resolve_path(cfg["apply"]["rag_predictions_csv"])
rag_predictions_csv.parent.mkdir(parents=True, exist_ok=True)

if rag_predictions_csv.exists():
    rag_df = pd.read_csv(rag_predictions_csv)
    print(f"Using existing RAG predictions: {rag_predictions_csv} ({len(rag_df)} rows)")
else:
    labeled_csv = resolve_path(cfg["feature_extraction"]["labeled_csv"])
    src = pd.read_csv(labeled_csv)
    if DEMO_LIMIT:
        src = src.head(int(DEMO_LIMIT)).copy()

    # Stand-in RAG output: corpus answer as predicted_answer, context as JSON list.
    demo = pd.DataFrame(
        {
            "id": src["id"],
            "question": src["question"],
            "gold_answer": src.get("answer", ""),
            "predicted_answer": src["answer"],
            "contexts": src["context"].apply(lambda c: json.dumps([str(c)], ensure_ascii=False)),
            "dataset": src.get("dataset", ""),
        }
    )
    demo.to_csv(rag_predictions_csv, index=False)
    rag_df = demo
    print(f"Created demo RAG predictions at {rag_predictions_csv} ({len(rag_df)} rows)")

rag_df.head(3)

## 2. Stage 1 — Build RAGAS features (labeled corpus)

Computes RAGAS metrics per `(question, context, answer)` row and saves `results/ragas_filter/ragas_features.csv`. Checkpointing resumes if interrupted.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is missing. Create a .env file in the repo root with:\n"
        "OPENAI_API_KEY=sk-..."
    )

ragas_cfg = cfg["ragas"]
fe_cfg = cfg["feature_extraction"]

labeled_csv = resolve_path(fe_cfg["labeled_csv"])
feature_path = resolve_path(fe_cfg["feature_path"])
checkpoint_path = resolve_path(fe_cfg["checkpoint_path"])
feature_path.parent.mkdir(parents=True, exist_ok=True)

labeled_df = pd.read_csv(labeled_csv)
if DEMO_LIMIT:
    labeled_df = labeled_df.head(int(DEMO_LIMIT)).copy()
print(f"Building features for {len(labeled_df)} labeled rows")

ragas = RAGAS(
    metrics=ragas_cfg["metrics"],
    llm_model=ragas_cfg["llm_model"],
    embedding_model=ragas_cfg["embedding_model"],
    temperature=ragas_cfg.get("temperature", 0.0),
)
extractor = RagasFeatureExtractor(
    ragas_evaluator=ragas,
    feature_cols=ragas_cfg["metrics"],
)

features_df = extractor.transform(
    data=labeled_df,
    feature_path=feature_path,
    checkpoint_path=checkpoint_path,
    batch_size=int(ragas_cfg.get("batch_size", 20)),
)
print("Saved:", feature_path)
features_df.head()

## 3. Stage 2 — Train RAGAS filter + select threshold

Trains several sklearn models on RAGAS feature columns, saves the best `.joblib`, then picks the **min-FPR threshold** subject to recall ≥ 0.70.

In [ ]:
tr_cfg = cfg["training"]

trainer = RagasFilterTrainer(
    feature_data=features_df,
    output_dir=str(resolve_path(tr_cfg["model_output"])),
    label_col=tr_cfg.get("label_col", "label"),
    id_col=tr_cfg.get("id_col", "id"),
    sort_by=tr_cfg.get("sort_by", "f1"),
    test_size=tr_cfg.get("test_size", 0.2),
    random_state=tr_cfg.get("random_state", 42),
)
train_out = trainer.run()

print("Best model:", train_out["best_model_name"])
print("Model path:", train_out["model_path"])
display(train_out["results_df"])

best_preds = train_out["test_predictions"]
best_preds = best_preds[best_preds["model"] == train_out["best_model_name"]]
threshold_result = select_threshold_min_fpr(
    confidences=best_preds["y_prob"].tolist(),
    labels=best_preds["y_true"].astype(int).tolist(),
    min_recall=tr_cfg.get("min_recall_for_threshold", 0.70),
)
threshold_path = resolve_path(tr_cfg["threshold_selection_path"])
threshold_path.parent.mkdir(parents=True, exist_ok=True)
with open(threshold_path, "w", encoding="utf-8") as fh:
    json.dump(threshold_result, fh, indent=2, default=str)

print(
    f"Selected threshold={threshold_result['threshold']:.3f} "
    f"(FPR={threshold_result['fpr']:.3f}, recall={threshold_result['recall']:.3f})"
)

## 4. Stage 3 — Apply filter to RAG predictions

Runs RAGAS feature extraction on generated answers, then accept/reject using the trained model and selected threshold.

In [ ]:
apply_cfg = cfg["apply"]
threshold = float(threshold_result["threshold"])

rag_for_filter = rag_df.copy()
if DEMO_LIMIT:
    rag_for_filter = rag_for_filter.head(int(DEMO_LIMIT)).copy()
# Corpus label does not describe generated answers -> inference-only.
rag_for_filter = rag_for_filter.drop(columns=["label"], errors="ignore")

ragas_filter = RagasFilter(
    model_path=train_out["model_path"],
    feature_extractor=extractor,
    output_dir=str(resolve_path(apply_cfg["results_dir"])),
    threshold=threshold,
)

filtered_df = ragas_filter.predict(
    data=rag_for_filter,
    feature_path=resolve_path(apply_cfg["feature_path"]),
    checkpoint_path=resolve_path(apply_cfg["checkpoint_path"]),
    filter_path=resolve_path(apply_cfg["filtered_csv"]),
    batch_size=int(ragas_cfg.get("batch_size", 20)),
)

n_accept = int((filtered_df["filter_label"] == 1).sum())
print(
    f"Accepted {n_accept}/{len(filtered_df)} "
    f"({100 * n_accept / max(len(filtered_df), 1):.1f}%) at threshold={threshold:.3f}"
)

answer_col = "predicted_answer" if "predicted_answer" in filtered_df.columns else "answer"
display(
    filtered_df[
        ["id", "question", answer_col, "filter_label", "filter_confidence"]
    ].head(10)
)

## 5. Results & outputs

| Artifact | Path |
|----------|------|
| Training features | `results/ragas_filter/ragas_features.csv` |
| Trained model | `models/ragas_filter/<best_model>.joblib` |
| Threshold selection | `results/ragas_filter/threshold_selection.json` |
| Filtered RAG output | `results/ragas_filter/predictions_filtered.csv` |

## 6. CLI alternative (headless)

From repo root with venv activated:

```powershell
# Full pipeline (recommended for reproducibility)
python scripts/run_ragas_pipeline.py --train-limit 20 --apply-limit 20

# Or stage by stage:
python scripts/build_ragas_features.py --limit 20
python scripts/train_ragas_filter.py
python scripts/run_ragas_filter_on_rag.py --limit 20
```

**Production run:** remove `--train-limit` / `--apply-limit` and set `DEMO_LIMIT = None` in this notebook.